In [87]:
import requests
import lxml.html
import time
import pandas as pd

In [43]:
def make_request(url):
    resp = requests.get(url)

    if resp.status_code == 429:
        time.sleep(2.5)
        resp = make_request(url)
    elif resp.status_code == 200:
        return resp

In [84]:
def scrape_case(case_url):
    url = "https://statecourtreport.org" + case_url
    root = lxml.html.fromstring(make_request(url).text)

    rd = {"docket_no": None, "title": None, "date": None, "type": None}

    xp1 = "//div[@class = 'field field--name-field-docket-number "
    xp2 = "field--type-string field--label-inline clearfix']"
    xp3 = "//div[@class = 'field__item']/text()"
    docket_no = root.xpath(xp1 + xp2 + xp3)[0]
    rd["docket_no"] = docket_no

    xp1 = "//div[@class = 'case-header__inner']"
    xp2 = "//div[@class = 'case-header__left']"
    xp3 = "//h1[@class = 'h1']//span/text()"
    title = root.xpath(xp1 + xp2 + xp3)[0]
    rd["title"] = title

    xp1 = "//div[@class = 'field field--name-field-date "
    xp2 = "field--type-datetime field--label-inline clearfix']"
    xp3 = "//div[@class = 'field__item']//time/@datetime"
    date = root.xpath(xp1 + xp2 + xp3)[0]
    rd["date"] = date

    xp1 = "//ul[@class = 'tags']//li[contains(@class, 'tags__item--primary')]"
    xp2 = "//a/text()"
    case_type = root.xpath(xp1 + xp2)
    if case_type == []:
        rd["type"] = None
    else:
        rd["type"] = case_type[0].replace("\n", "").strip()

    return rd

In [55]:
def scrape_page(url, rd):
    root = lxml.html.fromstring(make_request(url).text)

    xp1 = "//h2[@class = 'card__heading']"
    xp2 = "//a[@class = 'card__heading__link']/@href"
    case_links = root.xpath(xp1 + xp2)

    for link in case_links:
        case_info = scrape_case(link)

        for field in rd.keys():
            rd[field].append(case_info[field])

    return rd

In [62]:
def next_page_url(url):
    root = lxml.html.fromstring(make_request(url).text)

    rl = root.xpath("//a[@class = 'pager__link pager__link--next']/@href")

    if rl == []:
        return ""
    else:
        return rl[0]

In [71]:
def multi_page(start_url):
    url = start_url
    url_base = "https://statecourtreport.org/state-case-database"

    rd = {"docket_no": [], "title": [], "date": [], "type": []}

    while True:
        page_info = scrape_page(url, rd)

        url = url_base + next_page_url(url)
        rd = page_info

        if url == url_base:
            break

    return rd

In [83]:
def scrape_main(url):
    root = lxml.html.fromstring(make_request(url).text)
    url_base = "https://statecourtreport.org/state-case-database"

    xp1 = "//select[@id = 'edit-state']"
    xp2 = "//option[not(contains(@value, 'All'))]/@value"
    state_nos = root.xpath(xp1 + xp2)

    xp2 = "//option[not(contains(@value, 'All'))]/text()"
    state_names = root.xpath(xp1 + xp2)

    rd = {"docket_no": [], "state": [], "title": [], "date": [], "type": []}

    for i, state_code in enumerate(state_nos):
        length = 0
        state_url = url_base + f"?state={state_code}&issue=All&year=All"
        state_info = multi_page(state_url)

        for field in rd.keys():
            if field == "state":
                continue
            else:
                rd[field] += state_info[field]
                length = len(state_info[field])

        rd["state"] += [state_names[i]] * length

    return rd

In [85]:
cases_pd = scrape_main("https://statecourtreport.org/state-case-database")

In [88]:
pd.DataFrame(cases_pd)

,docket_no,state,title,date,type
0,425A21-3,Alabama,Hoke County Board of Education v. State of Nor...,2026-04-02T12:00:00Z,Education
1,3 WAP 2024,Alabama,Commonwealth v. Lee,2026-03-26T12:00:00Z,Criminal Law
2,SC101412,Alabama,Luther v. Hoskins,2026-03-24T12:00:00Z,Voting Rights and Elections
3,SC-2025-0372,Alabama,Jennings v. Smith,2026-03-13T12:00:00Z,Criminal Law
4,SC-2024-0081,Alabama,Crenshaw ex rel. Crenshaw v. Sonic Drive In of...,2024-12-06T12:00:00Z,Economic and Labor Rights
...,...,...,...,...,...
1600,S-22-0049,Wyoming,Villafana v. State,2022-10-25T12:00:00Z,Criminal Law
1601,425A21-3,Wyoming,Hoke County Board of Education v. State of Nor...,2026-04-02T12:00:00Z,Education
1602,3 WAP 2024,Wyoming,Commonwealth v. Lee,2026-03-26T12:00:00Z,Criminal Law
1603,SC101412,Wyoming,Luther v. Hoskins,2026-03-24T12:00:00Z,Voting Rights and Elections
